# Quantization intro

This notebook focuses on post-training quantization methods for Google Gemma 3 model with 270m parameters.

## Installation

In [1]:
%pip install -U transformers accelerate datasets huggingface_hub llmcompressor hqq bitsandbytes rouge-score

  Using cached huggingface_hub-2.3.0-py3-none-any.whl.metadata (16 kB)


In [ ]:
!hf download csebuetnlp/xlsum --repo-type dataset --revision refs/convert/parquet --include "ukrainian/**" --local-dir data/raw/xlsum

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0% 0/3 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/14.6M [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/29.1M [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/160M [00:00<?, ?B/s] 
Reconstructing (incomplete total...):   9% 14.6M/160M [00:01<00:10, 13.3MB/s, 1.93MB/s  ]

Reconstructing (incomplete total...):  18% 28.1M/160M [00:02<00:06, 20.3MB/s, 18.6MB/s  ]

Fetching 3 files:  67% 2/3 [00:02<00:01,  1.07s/it]
Fetching 3 files: 100% 3/3 [00:02<00:00,  1.23it/s]
Download complete: 100% 160M/160M [00:02<00:00, 140MB/s,  140MB/s  ]
Reconstruction complete: 100% 160M/160M [00:02<00:00, 182MB/s,  182MB/s  ]             ✓ Downloaded
  path: /content/data/raw/xlsum
Download complete: 100% 160M/160

In [2]:
import torch
import bitsandbytes as bnb
import gc
import numpy as np
import matplotlib.pyplot as plt
import json
import dotenv
import re
import unicodedata
import pandas as pd

from pathlib import Path
from getpass import getpass
from huggingface_hub import login
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier, QuantizationModifier
from compressed_tensors.offload import dispatch_model
from llmcompressor.modifiers.transform.awq import AWQModifier
from hqq.core.quantize import BaseQuantizeConfig
from hqq.models.hf.base import AutoHQQHFModel
from rouge_score import rouge_scorer
from sklearn.model_selection import ParameterGrid

dotenv.load_dotenv()

False

In [3]:
login(token=getpass("Hugging Face token: "), add_to_git_credential=False)


Hugging Face token: ··········


In [4]:
# device = "mps" if torch.backends.mps.is_available() else "cpu"
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "google/gemma-3-270m-it"
DTYPE = torch.float32
NUM_CALIBRATION_SAMPLES = 64
MAX_SEQ_LENGTH = 256

In [5]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

validation_dataset = load_dataset(
    "parquet",
    data_files={
        "validation": str(PROJECT_ROOT / "data" / "raw" / "xlsum" / "ukrainian" / "validation" / "0000.parquet")
    },
    split="validation",
)

val_subset = validation_dataset.shuffle(seed=42).select(range(20))
val_data = val_subset["text"]
val_summaries = val_subset["summary"]

In [6]:
class UnicodeWordTokenizer:
  def tokenize(self, text):
    text = unicodedata.normalize("NFC", text).lower()
    text = text.replace("’", "'").replace("ʼ", "'")
    return re.findall(r"[^\W_]+(?:'[^\W_]+)*", text)

rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=UnicodeWordTokenizer())

### Helper functions

In [7]:
def clear_memory():
    """
    Helper function to run garbage collection
    and release unused CUDA/MPS cached memory.
    """
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

def describe_model(model):
    """
    Helper function to retrieve information
    about model name and layers.
    """
    return type(model).__name__, model.model.layers

def model_size_mb(model):
    """
    Helper function to retrieve model size in MiB.
    """
    return model.get_memory_footprint() / 1_000_000

def load_baseline_model():
    """
    Helper function to load baseline model
    as a template for quantization methods.
    """
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(device).eval()


def tokenize_recieve(model, prompt):
    """
    Helper function to tokenize the prompt
    and recieve an answer.
    """
    message = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(message, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)

    input_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0, input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer

def xlsum_prompt_formation(record):
    record = re.sub(r" {2,}", " ", record)

    prompt = (
        "Підсумуй цю статтю:\n"
        f"{record}\n"
    )

    return prompt

def evaluate_model(model, tokenizer):
    """
    Helper function to evaluate model's loss
    """
    model.eval()
    device = next(model.parameters()).device

    total_loss, total_tokens = 0.0, 0

    for prompt in val_data:
        inputs = tokenizer(prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
        ).to(device)

        with torch.no_grad():
            outputs = model(
                **inputs,
                labels=inputs["input_ids"],
                use_cache=False,
            )

        token_count = inputs["input_ids"].shape[1] - 1
        total_loss += outputs.loss.item() * token_count
        total_tokens += token_count

    return { "loss": total_loss / total_tokens}

def evaluate_rouge(model):
    model.eval()
    rouge_res = []

    for text, reference in zip(val_data, val_summaries):
        prompt = xlsum_prompt_formation(text)
        prediction = tokenize_recieve(model, prompt)
        scores = rouge.score(target=reference, prediction=prediction)
        rouge_res.append({name: score.fmeasure for name, score in scores.items()})

    details = pd.DataFrame(rouge_res)
    rouge_averages = details[["rouge1", "rouge2", "rougeL"]].mean().to_dict()

    return rouge_averages

def model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_model_size, quantized_model, method_name):
    """
    Helper function for checking and comparing model's results.
    """
    example_prompt = "Explain quantization of Large Language Models in 1 sentence."

    loss_results_quantization = evaluate_model(quantized_model, tokenizer)
    loss_q = loss_results_quantization["loss"]

    rouge_results_quantization = evaluate_rouge(quantized_model)
    rouge1, rouge2, rougeL = rouge_results_quantization["rouge1"], rouge_results_quantization["rouge2"], rouge_results_quantization["rougeL"]

    print(f"\nBaseline model size (MB): {baseline_model_size:.3f}")
    print(f"Quantized {method_name} model size (MB): {model_size_mb(quantized_model):.3f}")

    print(f"Baseline loss: {baseline_model_loss:.4f}")
    print(f"{method_name} loss: {loss_q:.4f}\n")

    print("Baseline Rouge: "
          f"Rouge1: {baseline_model_rouge["rouge1"]:.4f}, "
          f"Rouge2: {baseline_model_rouge["rouge2"]:.4f}, "
          f"RougeL: {baseline_model_rouge["rougeL"]:.4f}")

    print(f"{method_name} Rouge: "
          f"Rouge1: {rouge1:.4f}, "
          f"Rouge2: {rouge2:.4f}, "
          f"RougeL: {rougeL:.4f}")

    print(f"\nPrompt: {example_prompt}")
    print(f"\nOutput Baseline: {tokenize_recieve(baseline_model, example_prompt).strip("\n")}")
    print(f"Output {method_name}: {tokenize_recieve(quantized_model, example_prompt).strip("\n")}")

## Baseline model

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(device)
model.eval()

model_type, baseline_model_descr = describe_model(model)

print(f"Model loaded on: {model.device}")
print(model_type)
print(baseline_model_descr)

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Model loaded on: cuda:0
Gemma3ForCausalLM
ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layerno

In [10]:
res = tokenize_recieve(model, "Explain machine learning in 1 simple sentence.")
print(res)

Machine learning is a type of artificial intelligence that uses algorithms to learn from data without being explicitly programmed.



In [11]:
baseline_model = load_baseline_model()
evaluation_results_baseline = evaluate_model(baseline_model, tokenizer)
baseline_model_loss = evaluation_results_baseline["loss"]
baseline_model_rouge = evaluate_rouge(baseline_model)
baseline_size = model_size_mb(baseline_model)

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

### Calibration Dataset preparation

In [12]:
c4_dataset = load_dataset("allenai/c4", "uk", split="train", streaming=True,)
calibration_texts = [row["text"] for row in c4_dataset.take(NUM_CALIBRATION_SAMPLES)]

encoded = tokenizer(calibration_texts, truncation=True, max_length=MAX_SEQ_LENGTH, padding=False, add_special_tokens=False)
calibration_dataset = Dataset.from_dict(dict(encoded))

print("Calibration samples:", len(calibration_dataset))
print(calibration_texts[0][:500])

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Calibration samples: 64
Як обмежити швидкість Інтернету? | Комп'ютер
Відповіді на питання. > Комп'ютер > Як обмежити швидкість Інтернету?
Як обмежити швидкість Інтернету?
Насправді, суть даної дії зовсім не в тому, щоб назавжди забезпечити себе слабкими швидкостями. Це необхідно лише для того, щоб розвантажити потік: коли швидкість використовується на повну котушку, нічого іншого з Інтернетом зробити просто не можна.
Якщо Ви володієте роутером, то Ви зможете обмежити швидкість всієї мережі в цілому, так і окремих прист


## Linear symmetric and asymmetric quantization

In [ ]:
def symmetric_quantize(x: torch.Tensor, dtype=torch.int8) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Function implements Linear Symmetric Quantization per tensor.
    Zero point is fixed at 0.

    :param x: torch.Tensor, single tensor
    :param dtype: torch.dtype, type to downcast tensor to
    :return: tuple[torch.Tensor, torch.Tensor], downcast tensor, scale
    """
    max_original = x.abs().max()
    max_int = torch.iinfo(dtype).max
    s = max_original / max_int
    s = torch.where(s == 0, torch.ones_like(s), s)
    q = torch.round(x / s).clamp(-max_int, max_int).to(dtype)

    return q, s

def asymmetric_quantize(x: torch.Tensor, dtype=torch.int8) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Function implements Linear Asymmetric Quantization per tensor.

    :param x: torch.Tensor, single tensor
    :param dtype: torch.dtype, type to downcast tensor to
    :return: tuple[torch.Tensor, torch.Tensor, torch.Tensor], downcast tensor, scale, zero point
    """
    min_original, max_original = x.min().clamp(max=0), x.max().clamp(min=0)
    min_int, max_int = torch.iinfo(dtype).min, torch.iinfo(dtype).max

    s = (max_original - min_original) / (max_int - min_int)
    s = torch.where(s == 0, torch.ones_like(s), s)

    z = torch.round(min_int - min_original / s).clamp(min_int, max_int).to(dtype)
    q = (torch.round(x / s) + z.float()).clamp(min_int, max_int).to(dtype)

    return q, s, z

def dequantize(q: torch.Tensor, s: torch.Tensor, z: int | torch.Tensor = 0) -> torch.Tensor:
    """
    Function implements Linear Dequantization per tensor.

    :param q: torch.Tensor, quantized tensor
    :param s: torch.Tensor, scale
    :param z: torch.Tensor, zero point
    :return: torch.Tensor, dequantized tensor
    """
    return s.float() * (q.float() - torch.as_tensor(z, dtype=torch.float32, device=q.device))


class QuantizedLinear(torch.nn.Module):
    @torch.no_grad()
    def __init__(self, linear, method="symmetric"):
        super().__init__()

        self.in_features = linear.in_features
        self.out_features = linear.out_features
        weights = linear.weight.detach().float()

        if method == "symmetric":
            q, scale = symmetric_quantize(weights)
            zero_point = torch.zeros((), dtype=torch.int8, device=q.device)
        elif method == "asymmetric":
            q, scale, zero_point = asymmetric_quantize(weights)

        self.register_buffer("bias", linear.bias.detach().clone() if linear.bias is not None else None)
        self.register_buffer("qweight", q)
        self.register_buffer("scale", scale)
        self.register_buffer("zero_point", zero_point)

    def forward(self, x):
        weight = dequantize(self.qweight, self.scale, self.zero_point).to(dtype=x.dtype)
        bias = (self.bias.to(dtype=x.dtype) if self.bias is not None else None)

        return torch.nn.functional.linear(x, weight, bias)


In [ ]:
@torch.no_grad()
def linear_weights_quantization(model, method="symmetric"):
    results = []

    _, model_layers = describe_model(model)

    def replace_layers(parent):
        for name, module in list(parent.named_children()):
            if isinstance(module, torch.nn.Linear):
                quantized = QuantizedLinear(module, method)

                results.append({
                    "layer": name,
                    "shape": tuple(quantized.qweight.shape),
                    "num_weights": quantized.qweight.numel(),
                    "scale": quantized.scale.item(),
                    "zero_point": quantized.zero_point.item(),
                })

                setattr(parent, name, quantized)
            else:
                replace_layers(module)

    replace_layers(model_layers)
    return results

In [43]:
symmetric_model = load_baseline_model()

results_sym = linear_weights_quantization(symmetric_model, method="symmetric",)
prompt_results_symmetric = model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_size, symmetric_model, "Linear Symmetric")

del symmetric_model
clear_memory()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]


Baseline model size (MB): 1072.395
Quantized Linear Symmetric model size (MB): 771.585
Baseline loss: 3.7793
Linear Symmetric loss: 3.9683

Baseline Rouge: Rouge1: 0.1037, Rouge2: 0.0268, RougeL: 0.0824
Linear Symmetric Rouge: Rouge1: 0.1070, Rouge2: 0.0263, RougeL: 0.0859

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output Linear Symmetric: Large Language Models (LLMs) are designed to process and understand natural language, achieving a state-of-the-art performance in various tasks.


In [44]:
asymmetric_model = load_baseline_model()

results_asym = linear_weights_quantization(asymmetric_model, method="asymmetric",)
prompt_results_asymmetric = model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_size, asymmetric_model, "Linear Asymmetric")

del asymmetric_model
clear_memory()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]


Baseline model size (MB): 1072.395
Quantized Linear Asymmetric model size (MB): 771.585
Baseline loss: 3.7793
Linear Asymmetric loss: 3.8854

Baseline Rouge: Rouge1: 0.1037, Rouge2: 0.0268, RougeL: 0.0824
Linear Asymmetric Rouge: Rouge1: 0.1024, Rouge2: 0.0206, RougeL: 0.0767

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output Linear Asymmetric: Quantization reduces the model's size and computational cost, making it suitable for lower-resource environments.


## NF4 model - bitsandbytes

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)

nf4_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quantization_config, device_map={"": 0}, dtype=DTYPE).eval()
prompt_results_nf4 = model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_size, nf4_model, "NF4")

del nf4_model
clear_memory()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]


Baseline model size (MB): 1072.395
Quantized NF4 model size (MB): 721.449
Baseline loss: 3.7793
NF4 loss: 4.3367

Baseline Rouge: Rouge1: 0.0250, Rouge2: 0.0000, RougeL: 0.0250
NF4 Rouge: Rouge1: 0.0250, Rouge2: 0.0000, RougeL: 0.0250

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output NF4: Quantization of Large Language Models (LLMs) aims to reduce the size and complexity of the model by lowering the number of parameters, typically to fractions of the original size. This reduces the model's computational cost, memory requirements, and training time, leading to improved performance on specific, often less-studied, tasks.


## GPTQ

In [46]:
gptq_model = load_baseline_model()
gptq_recipe = GPTQModifier(targets=["Linear"], scheme="W4A16", ignore=["lm_head"], dampening_frac=0.1, actorder=None)

oneshot(model=gptq_model, tokenizer=tokenizer, dataset=calibration_dataset, recipe=gptq_recipe,
        max_seq_length=MAX_SEQ_LENGTH,
        num_calibration_samples=len(calibration_dataset),
        batch_size=1,
        shuffle_calibration_samples=False)

dispatch_model(gptq_model)
prompt_results_gptq = model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_size, gptq_model, "GPTQ")

del gptq_model
clear_memory()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

2026-10-24T12:59:40.2838 | reset | INFO - Compression lifecycle reset
2026-10-24T12:59:40.2944 | norm_calibration_context | INFO - Found 109 offset-norm modules to convert
2026-10-24T12:59:40.3073 | from_modifiers | INFO - Creating recipe from modifiers


Applying quantization config: 100%|██████████| 126/126 [00:00<00:00, 4581.59it/s]

2026-10-24T12:59:40.3488 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-10-24T12:59:40.3502 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`



(2/19): Calibrating: 100%|██████████| 64/64 [00:00<00:00, 179.23it/s]

...

2026-10-24T12:59:52.3305 | compress_module_list | INFO - Quantizing 1 module(s): ['model.layers.17.self_attn.q_proj']


2026-10-24T12:59:52.3389 | GPTQ | METRIC - time 0.01s
2026-10-24T12:59:52.3396 | GPTQ | METRIC - error 4745.75
2026-10-24T12:59:52.3417 | GPTQ | METRIC - Accelerator 0 | usage: 94.10% | total memory: 15.6 Gb
2026-10-24T12:59:52.3450 | compress_module_list | INFO - Quantizing 2 module(s): ['model.layers.17.self_attn.k_proj', 'model.layers.17.self_attn.v_proj']
2026-10-24T12:59:52.3550 | GPTQ | METRIC - time 0.01s
2026-10-24T12:59:52.3556 | GPTQ | METRIC - error 1439.52
2026-10-24T12:59:52.3578 | GPTQ | METRIC - Accelerator 0 | usage: 94.10% | total memory: 15.6 Gb
2026-10-24T12:59:52.3605 | GPTQ | METRIC - time 0.01s
2026-10-24T12:59:52.3611 | GPTQ | METRIC - error 1541.62
2026-10-24T12:59:52.3635 | GPTQ | METRIC - Accelerator 0 | usage: 94.10% | total memory: 15.6 Gb
2026-10-24T12:59:52.3669 | compress_module_list | INFO - Quantizing 1 module(s): ['model.layers.17.self_attn.o_proj']
2026-10-24T12:59:52.3816 | GPTQ | METRIC - time 0.01s
2026-10-24T12:59:52.3823 | GPTQ | METRIC - error 5

(19/19): Propagating: 100%|██████████| 64/64 [00:00<00:00, 258.05it/s]

2026-10-24T12:59:52.6996 | norm_calibration_context | INFO - Restoring 109 norm modules to offset convention
2026-10-24T12:59:52.7101 | finalize | INFO - Compression lifecycle finalized for 1 modifiers



Baseline model size (MB): 1072.395
Quantized GPTQ model size (MB): 1076.312
Baseline loss: 3.7793
GPTQ loss: 4.3354

Baseline Rouge: Rouge1: 0.1037, Rouge2: 0.0268, RougeL: 0.0824
GPTQ Rouge: Rouge1: 0.0608, Rouge2: 0.0141, RougeL: 0.0536

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output GPTQ: Quantizing Large Language Models (LLMs) is a process of representing the information they process. This involves transforming the model's input and output into a different format, such as a vector representation. This transformation allows the model to perform different tasks and generate different responses. The goal is to make the model more accurate and efficient. This process involves training the model with different examples and ensuring that the model performs correctly. This is the core o

In [ ]:
param_grid = {"dampening_frac": [0.05, 0.1, 0.2], "actorder": [None, "static"],}
gptq_search_results = []

for params in ParameterGrid(param_grid):
  print(f"\nTesting: {params}", flush=True)

  gptq_model = load_baseline_model()

  try:
    recipe = GPTQModifier(targets=["Linear"], scheme="W4A16", ignore=["lm_head"], **params,)

    oneshot(model=gptq_model, tokenizer=tokenizer, dataset=calibration_dataset,
            recipe=recipe, max_seq_length=MAX_SEQ_LENGTH,
            num_calibration_samples=len(calibration_dataset),
            batch_size=1,
            shuffle_calibration_samples=False)

    dispatch_model(gptq_model)

    loss = evaluate_model(gptq_model, tokenizer)["loss"]
    rouge_scores = evaluate_rouge(gptq_model)

    gptq_search_results.append({**params, "loss": loss, "size_MB": model_size_mb(gptq_model), **rouge_scores})

  finally:
    del gptq_model
    clear_memory()

gptq_search_df = (pd.DataFrame(gptq_search_results).sort_values("loss").reset_index(drop=True))
display(gptq_search_df)


Testing: {'actorder': None, 'dampening_frac': 0.05}


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

2026-10-24T13:10:15.1032 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.
2026-10-24T13:10:15.2085 | reset | INFO - Compression lifecycle reset
2026-10-24T13:10:15.2438 | norm_calibration_context | INFO - Found 109 offset-norm modules to convert
2026-10-24T13:10:15.2571 | from_modifiers | INFO - Creating recipe from modifiers


Applying quantization config: 100%|██████████| 126/126 [00:00<00:00, 4786.46it/s]

2026-10-24T13:10:15.2949 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-10-24T13:10:15.2964 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


...

(19/19): Calibrating: 100%|██████████| 64/64 [00:00<00:00, 178.03it/s]

2026-10-24T13:18:30.6322 | compress_module_list | INFO - Quantizing 1 module(s): ['model.layers.17.self_attn.q_proj']
2026-10-24T13:18:30.6431 | GPTQ | METRIC - time 0.01s
2026-10-24T13:18:30.6438 | GPTQ | METRIC - error 6400.14
2026-10-24T13:18:30.6455 | GPTQ | METRIC - Accelerator 0 | usage: 68.51% | total memory: 15.6 Gb
2026-10-24T13:18:30.6506 | compress_module_list | INFO - Quantizing 2 module(s): ['model.layers.17.self_attn.k_proj', 'model.layers.17.self_attn.v_proj']
2026-10-24T13:18:30.6627 | GPTQ | METRIC - time 0.01s
2026-10-24T13:18:30.6634 | GPTQ | METRIC - error 1933.90
2026-10-24T13:18:30.6645 | GPTQ | METRIC - Accelerator 0 | usage: 68.51% | total memory: 15.6 Gb
2026-10-24T13:18:30.6661 | GPTQ | METRIC - time 0.01s
2026-10-24T13:18:30.6667 | GPTQ | METRIC - error 2069.49
2026-10-24T13:18:30.6677 | GPTQ | METRIC - Accelerator 0 | usage: 68.51% | total memory: 15.6 Gb
2026-10-24T13:18:30.6703 | compress_module_list | INFO - Quantizing 1 module(s): ['model.layers.17.self_

2026-10-24T13:18:30.6852 | GPTQ | METRIC - time 0.01s
2026-10-24T13:18:30.6858 | GPTQ | METRIC - error 694.21
2026-10-24T13:18:30.6871 | GPTQ | METRIC - Accelerator 0 | usage: 68.51% | total memory: 15.6 Gb
2026-10-24T13:18:30.6902 | compress_module_list | INFO - Quantizing 2 module(s): ['model.layers.17.mlp.gate_proj', 'model.layers.17.mlp.up_proj']
2026-10-24T13:18:30.7026 | GPTQ | METRIC - time 0.01s
2026-10-24T13:18:30.7035 | GPTQ | METRIC - error 614.15
2026-10-24T13:18:30.7044 | GPTQ | METRIC - Accelerator 0 | usage: 68.51% | total memory: 15.6 Gb
2026-10-24T13:18:30.7055 | GPTQ | METRIC - time 0.01s
2026-10-24T13:18:30.7062 | GPTQ | METRIC - error 628.18
2026-10-24T13:18:30.7074 | GPTQ | METRIC - Accelerator 0 | usage: 68.51% | total memory: 15.6 Gb
2026-10-24T13:18:30.7099 | compress_module_list | INFO - Quantizing 1 module(s): ['model.layers.17.mlp.down_proj']
2026-10-24T13:18:30.7416 | GPTQ | METRIC - time 0.03s
2026-10-24T13:18:30.7429 | GPTQ | METRIC - error 7.95
2026-10-24

(19/19): Propagating: 100%|██████████| 64/64 [00:00<00:00, 169.76it/s]

2026-10-24T13:18:31.1436 | norm_calibration_context | INFO - Restoring 109 norm modules to offset convention
2026-10-24T13:18:31.1590 | finalize | INFO - Compression lifecycle finalized for 1 modifiers


## AWQ

In [ ]:
awq_model = load_baseline_model()
awq_recipe = [AWQModifier(), QuantizationModifier(targets=["Linear"], scheme="W4A16_ASYM", ignore=["lm_head"])]

oneshot(model=awq_model, tokenizer=tokenizer, dataset=calibration_dataset, recipe=awq_recipe,
        max_seq_length=MAX_SEQ_LENGTH,
        num_calibration_samples=len(calibration_dataset),
        batch_size=1,
        shuffle_calibration_samples=False)

dispatch_model(awq_model)
prompt_results_awq = model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_size, awq_model, "AWQ")

del awq_model
clear_memory()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

2026-10-24T11:58:29.8174 | reset | INFO - Compression lifecycle reset
2026-10-24T11:58:29.8221 | norm_calibration_context | INFO - Found 109 offset-norm modules to convert
2026-10-24T11:58:29.8363 | from_modifiers | INFO - Creating recipe from modifiers
2026-10-24T11:58:29.8371 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...


Applying quantization config: 100%|██████████| 126/126 [00:00<00:00, 5100.74it/s]

2026-10-24T11:58:29.8734 | initialize | INFO - Compression lifecycle initialized for 2 modifiers
2026-10-24T11:58:29.8743 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `AWQModifier`
2026-10-24T11:58:29.9584 | _set_resolved_mappings | WARNING - 18 mappings were skipped due to incompatible shapes. Set logging to DEBUG to view full list.



(1/19): Calibrating: 100%|██████████| 64/64 [00:00<00:00, 357.74it/s]
Smoothing: 0it [00:00, ?it/s]
Smoothing:   0%|          | 0/2 [00:00<?, ?it/s]
Grid search for model.layers.0.pre_feedforward_layernorm:   0%|          | 0/20 [00:00<?, ?it/s]
Grid search for model.layers.0.pre_feedforward_layernorm:   0%|          | 0/20 [00:00<?, ?it/s, best_error=1.134e-02]
Grid search for model.layers.0.pre_feedforward_layernorm:   0%|          | 0/20 [00:00<?, ?it/s, best_error=9.846e-03]
Grid search for model.layers.0.pre_feedforward_layernorm:  10%|█         | 2/20 [00:00<00:01, 15.91it/s, best_error=9.846e-03]
Grid search for model.layers.0.pre_feedforward_layernorm:  10%|█         | 2/20 [00:00<00:01, 15.91it/s, best_error=9.201e-03]
Grid search for model.layers.0.pre_feedforward_layernorm:  10%|█         | 2/20 [00:00<00:01, 15.91it/s, best_error=9.201e-03]
Grid search for model.layers.0.pre_feedforward_layernorm:  20%|██        | 4/20 [00:00<00:00, 17.59it/s, best_error=9.201e-03]
Grid se

2026-10-24T11:59:10.1121 | IndependentPipeline | INFO - Inferred `DataFreePipeline` for `QuantizationModifier`


2026-10-24T11:59:10.2809 | norm_calibration_context | INFO - Restoring 109 norm modules to offset convention
2026-10-24T11:59:10.2977 | finalize | INFO - Compression lifecycle finalized for 2 modifiers

Baseline model size (MB): 1072.395
Quantized AWQ model size (MB): 1076.312
Baseline loss: 3.7793
AWQ loss: 4.4031

Baseline Rouge: Rouge1: 0.0250, Rouge2: 0.0000, RougeL: 0.0250
AWQ Rouge: Rouge1: 0.0250, Rouge2: 0.0000, RougeL: 0.0250

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output AWQ: **Quantization** is the process of lowering the precision of a Large Language Model (LLM) in a way that it can only represent a certain number of tokens (variables) in a fixed size. This allows the model to perform tasks that require a large amount of computation, such as reasoning, solving complex pro

## HQQ

In [ ]:
hqq_model = load_baseline_model()
quant_config = BaseQuantizeConfig(nbits=8, group_size=64)
AutoHQQHFModel.quantize_model(hqq_model, quant_config=quant_config, compute_dtype=DTYPE, device="cuda:0",)
prompt_results_hqq = model_results(baseline_model, baseline_model_loss, baseline_model_rouge, baseline_size, hqq_model, "HQQ")

del hqq_model
clear_memory()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

100%|██████████| 127/127 [00:00<00:00, 247.53it/s]



Baseline model size (MB): 1072.395
Quantized HQQ model size (MB): 771.585
Baseline loss: 3.7793
HQQ loss: 3.7935

Baseline Rouge: Rouge1: 0.0250, Rouge2: 0.0000, RougeL: 0.0250
HQQ Rouge: Rouge1: 0.0250, Rouge2: 0.0000, RougeL: 0.0250

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output HQQ: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
